## Packages
##### Data Analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, pointbiserialr, chi2_contingency, ttest_ind
from collections import Counter

##### Feature Selection

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

##### Model Building

In [ ]:
from statsmodels.api import OLS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

##### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

##### Text Processing

In [ ]:
import re
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tuckerparon/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Functions
##### get_sentiment_score()
Retrieve the sentiment score of a text string.

In [ ]:
def get_sentiment_score(text):
    try:
        # Remove stop words from the text
        words = text.split()
        filtered_words = [word for word in words if word.lower() not in stopwords.words()]
        filtered_text = ' '.join(filtered_words)

        # Calculate the sentiment score of the
        blob = TextBlob(filtered_text)
        return blob.sentiment.polarity
    except:
        return 101

##### get_sentiment_score_by_section()
Split 'Notes' into subsections based on emoji indicators and pass each section into get_sentiment_score. An example of value in 'Notes' is:

                ☀️Had a great time today at the waterpark. Met up with some friends and ate some really
                good food. ☁️ The drive home was really long, and we gota flat tire. 🎯 Finish up my
                CSYS 387 final project and enjoy the weather!

In [1]:
def get_sentiment_score_by_section(text):
    try:
        # remove newlines, split text in a list of words, and encode
        text = text.replace('\\n',' ').split()
        text = [word.encode('unicode_escape').decode() for word in text]

        # get index of sun, cloud, and target emojis
        sun_pos = text.index('\\u2600\\ufe0f')
        cloud_pos = text.index('\\u2601\\ufe0f')
        target_pos = text.index('\\U0001f3af')
        book_pos = text.index('\\U0001f4d6')

        # use indexes to divide text into subsections (good, bad, goals)
        good_text = text[sun_pos:cloud_pos]
        bad_text = text[cloud_pos:target_pos]
        goal_text = text[target_pos:book_pos]
        #book_text = text[book_text:]

        # decode the sections back to Unicode and combine words back into a string
        good_text = ' '.join([word.encode().decode('unicode_escape') for word in good_text])
        bad_text = ' '.join([word.encode().decode('unicode_escape') for word in bad_text])
        goal_text = ' '.join([word.encode().decode('unicode_escape') for word in goal_text])
        #book_text = ' '.join([word.encode().decode('unicode_escape') for word in book_text])

        # calculate the sentiment scores of each section
        good_score = get_sentiment_score(good_text)
        bad_score = get_sentiment_score(bad_text)
        goal_score = get_sentiment_score(goal_text)

        return good_score, bad_score, goal_score
    except:
        return None, None, None

##### custom_corr()
Retrieve the correlation between two variables using the appropriate method based on the variable types.
- If both variables are continuous, Pearson is appropriate because it assumes a linear relationship and a normal distribution
- If both variables are binary, the chi squared contingency coefficient is appropriate because it is specifically designed for categorical variables.
- If one variable is binary and the other continuous, point biserial is appropriate because it doesn't assume linear relationships or normality.


In [ ]:
# Define a custom correlation function to use pointbiserialr for binary-continuous variable pairs, contingency for binary pairs, and pearson for continuos pairs
def custom_corr(x, y):
    if len(set(x)) == 2 and len(set(y)) == 2: # if both binary
        c, p, dof, expected = chi2_contingency(pd.crosstab(x, y))
        return c/1000
    elif len(set(x)) == 2 and len(set(y)) > 2: # if one binary
        c, p = pointbiserialr(x, y)
        return c
    elif len(set(x)) > 2 and len(set(y)) == 2: # if one binary
        c, p = pointbiserialr(y, x)
        return c
    else:
        c, p = pearsonr(x, y) # if both continuous
        return c

##### custom_p()
Retrieve the p-value between two variables using the appropriate method based on the variable types. See previous function description for reasoning.

In [ ]:
def custom_p(x, y):
    if len(set(x)) == 2 and len(set(y)) == 2:
        c, p, dof, expected = chi2_contingency(pd.crosstab(x, y))
    elif len(set(x)) == 2 and len(set(y)) > 2:
        c, p = pointbiserialr(x, y)
    elif len(set(x)) > 2 and len(set(y)) == 2:
        c, p = pointbiserialr(y, x)
    else:
        c, p = pearsonr(x, y)
    return p

### Other
Large data cleaning variable.

In [ ]:
# clean tag names
appropriate_column_names = {
    'Different Sleeping Arrangement' : 'Different sleeping arrangement',
    'Hungover': 'Veisalgia',
    'Lots of water (before bed)': 'Hydration (20+ ounces, late night)',
    'Late Night Screen Time':'Screen time (late night)',
    'Masterbated (before reading)' : 'Autoeroticism (before reading)',
    'Masterbated (morning)' : 'Autoeroticism (morning)',
    'Masterbated (day)' : 'Autoeroticism (day)',
    'Masterbated (night)' : 'Autoeroticism (night, late night)',
    'Headache (mild)' : 'Headache',
    'Gluteus Medius' : 'Soreness (gluteus medius)',
    'Low Back' : 'Soreness (low back)',
    'Low Back / Gluteus Medius' : 'Soreness (low back, gluteus medius)',
    'Match' : 'Participated in soccer match',
    'Meditated (morning)' : 'Meditation (morning)',
    'Meditated (day)' : 'Meditation (day)',
    'Meditated (night)' : 'Meditation (night)',
    'Meditating' : 'Meditating (during reading)',
    'Mushrooms (micro)' : 'Psilocybin',
    'Need to pee' : 'Urinary urgency (during reading)',
    'Need to poop' : 'Bowel urgency (during reading)',
    'New Living Situation': 'Different living arrangement',
    'Poor eating' : 'Malnutrition',
    'Reading' : 'Reading (during reading)',
    'Reading (before bed)' : 'Reading (late night)',
    'Sex (day)' : 'Sexual intercourse (morning, day)',
    'Sex (night)' : 'Sexual intercourse (night, late night)',
    'Sick' : 'Ill',
    'Sitting Down A Lot': 'Sedentary (7+ hours)',
    'Slept Past Alarm' : 'Overslept',
    'Smoked Cigarette' : 'Tobacco (smoking)',
    'Snacking (day)' : 'Unplanned eating (day)',
    'Sore hamstrings' : 'Soreness (hamstring)',
    'Stomach ache' : 'Gastric pain',
    'Talking':'Dialogue (during reading)',
    'Up and about' : 'Up and about (before reading)',
    'Up early' : 'Early wake-up',
    'Up late' : 'Late bed-time',
    'Watching reading':'Observing reading (during reading)',
    'Watching TV' : 'Television (during reading)',
    'Weed (before bed)' : 'Cannabis (ingestion, smoking; night, late night)',
    'Weed (not before bed)' : 'Cannabis (ingestion, smoking; morning, day)',
    'Wet Dream': 'Nocturnal emission',
    'w/ Nala':'Canine co-sleeping (Nala)',
    'w/ Waldo':'Canine co-sleeping (Waldo)'}